In [0]:
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder \
    .appName("MySparkApp") \
    .master("local[*]") \
    .getOrCreate()

# Access SparkContext from SparkSession
sc = spark.sparkContext

In [0]:
df= spark.read.format('csv').option('inferSchema',True).option('header',True).load('/users/shubh/Data Engineering/BigMart Sales.csv')

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
%config Completer.use_jedi = False

In [0]:
df.toPandas()

# when otherwise

### Scenario 1

In [0]:
df= df.withColumn('Non_veg_flag', when(col('Item_Type')=='Meat','Non-veg').otherwise('Veg'))

In [0]:
df.show()

In [0]:
df.withColumn('Veg_exp_flag',when((col('Non_veg_flag')=='Veg') & (col('Item_MRP') < 100), 'Veg_Inexpensive')\
                 .when((col('Non_veg_flag')=='Veg') & (col('Item_MRP') > 100), 'Veg_expensive')\
                 .otherwise('Non-Veg')).toPandas()

# Joins

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

In [0]:
df1.show()
df2.show()

### inner join

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'],'inner').show()

### left join

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'],'left').show()

# right join

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'],'right').show()

### anti join

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'],'anti').show()

In [0]:
df2.join(df1, df2['dept_id']==df1['dept_id'],'anti').show()

# Window functions

### RowNumber()

In [0]:
from pyspark.sql.window import Window

In [0]:
df.withColumn('RowNumber',row_number().over(Window.orderBy('Item_Identifier'))).toPandas()

### Rank


In [0]:
df.withColumn('Rank',rank().over(Window.orderBy('Item_Identifier'))).show()

### DenseRank

In [0]:
df.withColumn('DenseRank', dense_rank().over(Window.orderBy('Item_Identifier'))).show()

In [0]:
df.withColumn('Rank',rank().over(Window.orderBy('Item_Identifier')))\
    .withColumn('DenseRank', dense_rank().over(Window.orderBy('Item_Identifier'))).show()  #combination of rank and DenseRank

# cumilative sum

In [0]:
df.withColumn('Cumsum', sum('Item_MRP').over(Window.orderBy('Item_Type'))).toPandas()

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type')\
                                            .rowsBetween(Window.unboundedPreceding,Window.currentRow))).show()

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type')\
                                            .rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))).show()

# User defined functions(UDF)

### Step1

In [0]:
def newFunc(x):
    return x*x

In [0]:
my_udf= udf(newFunc)

### Step2

In [0]:
df.withColumn('SquareMRP',my_udf('Item_MRP')).show()

# Data Writing

### csv

In [0]:
df.write.format('csv')\
        .save('/users/shubh/BigData_Data/Data.csv')

#### APPEND

In [0]:
df.write.format('csv')\
        .mode('append')\
        .save('/users/shubh/BigData_Data/Data.csv')

In [0]:
df.write.format('csv')\
        .mode('append')\
        .option('path','/users/shubh/BigData_Data/Data.csv')\
        .save()

#### Overwrite

In [0]:
df.write.format('csv')\
.mode('overwrite')\
.option('path','/users/shubh/BigData_Data/Data.csv')\
.save()

#### Error

In [0]:
df.write.format('csv')\
.mode('error')\
.option('path','/users/shubh/BigData_Data/Data.csv')\
.save()


#### PARQUET

In [0]:
df.write.format('parquet')\
.mode('overwrite')\
.option('path','/users/shubh/BigData_Data/Data.csv')\
.save()

#### TABLE

In [0]:
df.write.format('parquet')\
.mode('overwrite')\
.saveAsTable('my_table')

In [0]:
df.toPandas()


# SQL view

In [0]:
df.createTempView('my_view')

In [0]:
result_all = spark.sql("select * from my_view where Item_Fat_Content = 'Low Fat'")
result_all.show()

# End
